# Fink/LSST — Colour-Colour Diagram: (G−R) vs (R−I)

This notebook reads the object-level summary produced by **`05_fink_download_objects.ipynb`**
(`data_FINK_BLOCK_LC_01/objects_all.parquet`) and plots a **colour-colour diagram**
in AB magnitudes:

- **X-axis**: (R − I) magnitude
- **Y-axis**: (G − R) magnitude
- **Error bars** on both axes, propagated from per-band magnitude uncertainties

Objects are colour-coded by their classification `group` inherited from notebook 01.

## Data flow

```
data_FINK_BLOCK_LC_01/objects_all.parquet   (produced by notebook 05)
    ↓
median flux per band (g, r, i)  →  AB magnitude conversion
    ↓
G−R  vs  R−I  colour-colour diagram  (with error bars)
    ↓
figs_FINK_BLOCK_LC_01/colour_colour_gr_ri.png
```

## Flux → Magnitude conversion (AB system)

LSST/Fink fluxes are in **nanojansky (nJy)**.  
Conversion to AB magnitude:

$$m_{AB} = -2.5 \log_{10}(f_{\rm nJy}) + 31.4$$

Uncertainty propagation:

$$\sigma_m = \frac{2.5}{\ln 10} \cdot \frac{\sigma_f}{f}$$

Colour uncertainties (quadratic sum):

$$\sigma_{G-R} = \sqrt{\sigma_{m_g}^2 + \sigma_{m_r}^2}, \quad
\sigma_{R-I} = \sqrt{\sigma_{m_r}^2 + \sigma_{m_i}^2}$$


## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")
print(f"matplotlib version : {plt.matplotlib.__version__}")

In [ ]:
# ── Directory & file paths (must match notebook 05 configuration) ─────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}_01_02"  # output figure directory (already exists)

PARQUET_OBJECTS = os.path.join(DIR_DATA, "objects_all.parquet")
OUT_FIG_PNG = os.path.join(DIR_FIGS, "colour_colour_gr_ri.png")
OUT_FIG_PDF = os.path.join(DIR_FIGS, "colour_colour_gr_ri.pdf")

os.makedirs(DIR_FIGS, exist_ok=True)

print(f"Input parquet : {os.path.abspath(PARQUET_OBJECTS)}")
print(f"Output figure : {os.path.abspath(OUT_FIG_PNG)}")

assert os.path.exists(PARQUET_OBJECTS), (
    f"{PARQUET_OBJECTS} not found.\nPlease run notebook 05_fink_download_objects.ipynb first."
)

## 2. Load `objects_all.parquet` and inspect available columns

In [ ]:
df = pd.read_parquet(PARQUET_OBJECTS)

print(f"DataFrame shape : {df.shape}")
print(f"Unique objects  : {df['diaObjectId'].nunique()}")
print(f"Groups present  : {sorted(df['group'].unique())}")
print(f"\nAll columns ({len(df.columns)}) :")
for col in sorted(df.columns):
    print(f"  {col}")

In [ ]:
# Inspect sample rows to identify per-band photometric columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 260)
df.head(3)

## 3. Identify per-band flux/magnitude columns

The Fink `/api/v1/objects` endpoint returns aggregate (median/mean) photometry per band.
Column naming conventions observed in the LSST Fink portal:

| Quantity | Possible column names |
|---|---|
| Median flux (nJy) | `o:medianFluxG`, `o:medianFluxR`, `o:medianFluxI` |
| Flux uncertainty (nJy) | `o:sigmaFluxG`, `o:sigmaFluxR`, `o:sigmaFluxI` |
| Median magnitude | `o:medianMagG`, `o:medianMagR`, `o:medianMagI` |
| Magnitude uncertainty | `o:sigmaMagG`, `o:sigmaMagR`, `o:sigmaMagI` |

The cell below performs **automatic column discovery** — it searches for any column
containing band identifiers and flux/magnitude keywords, then falls back to a manual
conversion from the `flatness_metrics.csv` mean fluxes if needed.

In [ ]:
def find_band_column(df_columns, band, keywords):
    """
    Search for a column matching a photometric band and a list of keywords.
    Returns the first matching column name, or None if not found.
    """
    b = band.lower()
    B = band.upper()
    for col in df_columns:
        col_lower = col.lower()
        if any(kw.lower() in col_lower for kw in keywords):
            if col.endswith(B) or col.endswith(b) or f"_{B}" in col or f"_{b}" in col:
                return col
    return None


cols = df.columns.tolist()

# Priority order: flux columns (nJy) → compute magnitude ourselves for full control
flux_keywords = ["medianflux", "median_flux", "meanflux", "mean_flux", "medflux", "flux_median", "flux_mean"]
flux_err_keywords = ["sigmaflux", "sigma_flux", "fluxerr", "flux_err", "errflux", "rmsflux", "rms_flux"]
mag_keywords = ["medianmag", "median_mag", "meanmag", "mean_mag", "magmedian", "mag_median"]
magerr_keywords = ["sigmamag", "sigma_mag", "magerr", "mag_err", "errmag", "rmsmag", "rms_mag"]

band_cols = {}
for band in ["g", "r", "i"]:
    band_cols[band] = {
        "flux": find_band_column(cols, band, flux_keywords),
        "fluxerr": find_band_column(cols, band, flux_err_keywords),
        "mag": find_band_column(cols, band, mag_keywords),
        "magerr": find_band_column(cols, band, magerr_keywords),
    }

print("Column discovery results:")
for band, d in band_cols.items():
    print(f"  band={band} : {d}")

## 4. Build per-object photometry table (flux → AB magnitude)

### Strategy
1. **If flux columns were found**: use them to compute AB magnitudes and uncertainties.
2. **If magnitude columns were found directly**: use them as-is.
3. **Fallback**: pivot `flatness_metrics.csv` (mean flux per band per object)
   and convert to magnitude — this covers the case where `objects_all.parquet`
   does not contain per-band photometry columns.

### AB magnitude convention (LSST/Fink, flux in nJy)
$$m = -2.5 \log_{10}(f_{\rm nJy}) + 31.4$$

In [ ]:
NJY_ZP = 31.4  # AB zero-point for fluxes in nJy: m = -2.5*log10(f_nJy) + 31.4


def flux_nJy_to_mag(flux, fluxerr=None):
    """
    Convert flux in nJy to AB magnitude.
    Returns (mag, magerr) where magerr=NaN if fluxerr is None.
    Invalid (negative or zero) flux values are masked as NaN.
    """
    flux = np.asarray(flux, dtype=float)
    valid = flux > 0
    mag = np.full_like(flux, np.nan)
    mag[valid] = -2.5 * np.log10(flux[valid]) + NJY_ZP

    if fluxerr is not None:
        fe = np.asarray(fluxerr, dtype=float)
        magerr = np.full_like(flux, np.nan)
        magerr[valid] = (2.5 / np.log(10)) * np.abs(fe[valid] / flux[valid])
    else:
        magerr = np.full_like(flux, np.nan)
    return mag, magerr


# Correct zero-point constant name
NJY_ZP = NJY_ZP  # already defined above


# ── Determine which strategy to use ──────────────────────────────────────────
has_flux_g = band_cols["g"]["flux"] is not None
has_flux_r = band_cols["r"]["flux"] is not None
has_flux_i = band_cols["i"]["flux"] is not None
has_mag_g = band_cols["g"]["mag"] is not None
has_mag_r = band_cols["r"]["mag"] is not None
has_mag_i = band_cols["i"]["mag"] is not None

strategy = "unknown"

if has_flux_g and has_flux_r and has_flux_i:
    strategy = "flux_columns"
elif has_mag_g and has_mag_r and has_mag_i:
    strategy = "mag_columns"
else:
    strategy = "fallback_flatness_csv"

print(f"Selected strategy: {strategy}")

In [ ]:
# ── Build the photometry DataFrame (one row per object) ───────────────────────

if strategy == "flux_columns":
    # --- Strategy A: compute magnitudes from flux columns --------------------
    print("Strategy A: flux columns found in objects_all.parquet")
    phot = df[["diaObjectId", "group"]].copy()

    for band in ["g", "r", "i"]:
        fc = band_cols[band]["flux"]
        fec = band_cols[band]["fluxerr"]
        flux = df[fc].values
        ferr = df[fec].values if fec else None
        mag, magerr = flux_nJy_to_mag(flux, ferr)
        phot[f"mag_{band}"] = mag
        phot[f"magerr_{band}"] = magerr
        print(f"  band={band}: flux col={fc}, fluxerr col={fec}")

elif strategy == "mag_columns":
    # --- Strategy B: magnitude columns already available --------------------
    print("Strategy B: magnitude columns found directly in objects_all.parquet")
    phot = df[["diaObjectId", "group"]].copy()

    for band in ["g", "r", "i"]:
        mc = band_cols[band]["mag"]
        mec = band_cols[band]["magerr"]
        phot[f"mag_{band}"] = df[mc].values.astype(float)
        phot[f"magerr_{band}"] = df[mec].values.astype(float) if mec else np.nan
        print(f"  band={band}: mag col={mc}, magerr col={mec}")

else:
    # --- Strategy C: fallback — use flatness_metrics.csv mean fluxes --------
    print("Strategy C (fallback): using flatness_metrics.csv mean fluxes")
    print("  Per-band photometry not found in objects_all.parquet.")
    print("  Using mean_flux_nJy from flatness_metrics.csv as proxy.")

    DIR_DATA_CSV = DIR_DATA
    flat_csv = os.path.join(DIR_DATA_CSV, "flatness_metrics.csv")
    assert os.path.exists(flat_csv), f"{flat_csv} not found."

    df_flat = pd.read_csv(flat_csv)
    df_flat["diaObjectId"] = df_flat["diaObjectId"].astype(str)

    # Pivot to get one row per object with columns mean_flux_g, mean_flux_r, mean_flux_i
    piv_flux = df_flat.pivot_table(
        index="diaObjectId", columns="band", values="mean_flux_nJy", aggfunc="first"
    ).reset_index()
    piv_flux.columns.name = None
    piv_flux.rename(columns={"g": "flux_g", "r": "flux_r", "i": "flux_i"}, inplace=True)

    # RMS variability as proxy for flux uncertainty
    piv_rms = df_flat.pivot_table(
        index="diaObjectId", columns="band", values="rms_var", aggfunc="first"
    ).reset_index()
    piv_rms.columns.name = None
    piv_rms.rename(columns={"g": "rms_g", "r": "rms_r", "i": "rms_i"}, inplace=True)

    # Group mapping
    group_map = df_flat[["diaObjectId", "group"]].drop_duplicates(subset="diaObjectId")

    phot = piv_flux.merge(piv_rms, on="diaObjectId").merge(group_map, on="diaObjectId")

    for band in ["g", "r", "i"]:
        flux = phot[f"flux_{band}"].values
        # rms_var is sigma/mean, so sigma_flux = rms_var * mean_flux
        fluxerr = phot[f"rms_{band}"].values * flux
        mag, magerr = flux_nJy_to_mag(flux, fluxerr)
        phot[f"mag_{band}"] = mag
        phot[f"magerr_{band}"] = magerr

print(f"\nPhotometry table: {len(phot)} rows")
phot[["diaObjectId", "group", "mag_g", "magerr_g", "mag_r", "magerr_r", "mag_i", "magerr_i"]].head(8)

## 5. Compute colours and colour uncertainties

In [ ]:
# Colour indices
phot["GR"] = phot["mag_g"] - phot["mag_r"]  # G − R
phot["RI"] = phot["mag_r"] - phot["mag_i"]  # R − I

# Colour uncertainties (quadratic sum of per-band magnitude uncertainties)
phot["eGR"] = np.sqrt(phot["magerr_g"] ** 2 + phot["magerr_r"] ** 2)
phot["eRI"] = np.sqrt(phot["magerr_r"] ** 2 + phot["magerr_i"] ** 2)

# Keep only rows with valid, finite colour measurements
mask_valid = (
    np.isfinite(phot["GR"]) & np.isfinite(phot["RI"]) & np.isfinite(phot["eGR"]) & np.isfinite(phot["eRI"])
)
phot_valid = phot[mask_valid].copy().reset_index(drop=True)

n_total = len(phot)
n_valid = len(phot_valid)
print(f"Objects total          : {n_total}")
print(f"Objects with valid g,r,i colours : {n_valid} ({100 * n_valid / n_total:.1f}%)")
print(f"Dropped (NaN/Inf)      : {n_total - n_valid}")
print()
print("Colour statistics (valid objects):")
print(phot_valid[["GR", "RI", "eGR", "eRI"]].describe().round(4))

In [ ]:
# Distribution of objects per group in the valid sample
print("Valid objects per group:")
print(phot_valid["group"].value_counts().to_string())

## 6. Colour-colour diagram: (G−R) vs (R−I)

In [ ]:
# ── Colour palette — one distinct colour per group ────────────────────────────
GROUP_COLOURS = {
    "gaia_star_stable": "royalblue",
    "gaia_star_variable": "dodgerblue",
    "vsx_variable": "darkorange",
    "tns_transient": "crimson",
    "simbad_galaxy": "mediumseagreen",
    "simbad_AG?": "goldenrod",
    "simbad_rG": "teal",
    "mangrove_galaxy_2mass": "mediumpurple",
    "unclassified": "silver",
}
DEFAULT_COLOUR = "black"

# Marker size and alpha — scale down if many objects
n = len(phot_valid)
MS = max(2.5, min(5, 300 / max(n, 1)))
ALPHA = max(0.3, min(0.85, 200 / max(n, 1)))

print(f"Plotting {n} objects   marker_size={MS:.1f}  alpha={ALPHA:.2f}")

In [ ]:
# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

groups = sorted(phot_valid["group"].unique())

for grp in groups:
    sub = phot_valid[phot_valid["group"] == grp]
    colour = GROUP_COLOURS.get(grp, DEFAULT_COLOUR)

    ax.errorbar(
        sub["RI"],
        sub["GR"],
        xerr=sub["eRI"],
        yerr=sub["eGR"],
        fmt="o",
        ms=MS,
        color=colour,
        ecolor=colour,
        alpha=ALPHA,
        elinewidth=0.5,
        capsize=1.5,
        capthick=0.5,
        label=f"{grp}  (n={len(sub)})",
        zorder=3,
    )

# ── Axes labels & formatting ──────────────────────────────────────────────────
ax.set_xlabel(r"$R - I$ (AB mag)", fontsize=14)
ax.set_ylabel(r"$G - R$ (AB mag)", fontsize=14)
ax.set_title(
    r"Colour-Colour Diagram: $(G-R)$ vs $(R-I)$" + "\nFink/LSST — Deep Drilling Fields",
    fontsize=14,
)

ax.tick_params(axis="both", labelsize=11)
ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-3.0, 3.0)

# ── Legend (outside the plot area on the right) ───────────────────────────────
legend = ax.legend(
    title="Object group",
    fontsize=9,
    title_fontsize=10,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    borderaxespad=0,
    framealpha=0.9,
    edgecolor="grey",
    handlelength=1.2,
    markerscale=1.5,
)

# ── Annotation: number of valid objects ───────────────────────────────────────
ax.annotate(
    f"N = {n} objects",
    xy=(0.02, 0.97),
    xycoords="axes fraction",
    va="top",
    ha="left",
    fontsize=10,
    color="dimgrey",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7, ec="none"),
)

plt.tight_layout()

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig(OUT_FIG_PNG, dpi=150, bbox_inches="tight")
fig.savefig(OUT_FIG_PDF, bbox_inches="tight")
print(f"Saved: {OUT_FIG_PNG}")
print(f"Saved: {OUT_FIG_PDF}")

plt.show()

## 7. Per-group colour statistics

In [ ]:
# Summary statistics of colours per group
stats = (
    phot_valid.groupby("group")[["GR", "RI", "eGR", "eRI"]].agg(["count", "mean", "median", "std"]).round(4)
)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 300)
print(stats.to_string())
stats

## 8. Magnitude histograms per band (diagnostic)

Quick sanity check: distributions of the g, r, i magnitudes entering the colour computation.

In [ ]:
fig2, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)

band_labels = {"mag_g": ("g", "steelblue"), "mag_r": ("r", "tomato"), "mag_i": ("i", "goldenrod")}

for ax_i, (col, (blabel, bcolor)) in zip(axes, band_labels.items()):
    vals = phot_valid[col].dropna()
    # Remove extreme outliers for display purposes (keep 1%–99% range)
    lo, hi = np.percentile(vals, [1, 99])
    vals_clipped = vals[(vals >= lo) & (vals <= hi)]

    ax_i.hist(vals_clipped, bins=40, color=bcolor, edgecolor="white", linewidth=0.4, alpha=0.85)
    ax_i.set_xlabel(f"{blabel.upper()} magnitude (AB)", fontsize=12)
    ax_i.set_ylabel("Number of objects", fontsize=11)
    ax_i.set_title(f"Band {blabel.upper()}   median={np.median(vals):.2f}", fontsize=12)
    ax_i.tick_params(labelsize=10)
    ax_i.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)

fig2.suptitle("Magnitude distributions per band (valid objects)", fontsize=13, y=1.01)
plt.tight_layout()

out_hist_png = os.path.join(DIR_FIGS, "magnitude_histograms_gri.png")
fig2.savefig(out_hist_png, dpi=150, bbox_inches="tight")
print(f"Saved: {out_hist_png}")
plt.show()

## 9. Colour-colour diagram — split panel per group

Each group is shown in its own sub-panel to avoid crowding when many objects overlap.

In [ ]:
groups_with_data = [
    g for g in sorted(phot_valid["group"].unique()) if len(phot_valid[phot_valid["group"] == g]) > 0
]
n_groups = len(groups_with_data)

# Determine grid layout
ncols = min(3, n_groups)
nrows = int(np.ceil(n_groups / ncols))

# Global colour axis limits (same for all panels)
ri_all = phot_valid["RI"]
gr_all = phot_valid["GR"]
p1, p99 = 1, 99
xlim = (np.percentile(ri_all, p1) - 0.1, np.percentile(ri_all, p99) + 0.1)
ylim = (np.percentile(gr_all, p1) - 0.1, np.percentile(gr_all, p99) + 0.1)

fig3, axes3 = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), sharex=True, sharey=True)
axes3_flat = np.atleast_1d(axes3).flatten()

for idx, grp in enumerate(groups_with_data):
    ax = axes3_flat[idx]
    sub = phot_valid[phot_valid["group"] == grp]
    colour = GROUP_COLOURS.get(grp, DEFAULT_COLOUR)

    # Background: all other objects in light grey
    rest = phot_valid[phot_valid["group"] != grp]
    ax.scatter(rest["RI"], rest["GR"], s=4, color="lightgrey", alpha=0.4, zorder=1, label="other")

    # Foreground: this group with error bars
    ax.errorbar(
        sub["RI"],
        sub["GR"],
        xerr=sub["eRI"],
        yerr=sub["eGR"],
        fmt="o",
        ms=max(5, MS),
        color=colour,
        ecolor=colour,
        alpha=min(ALPHA + 0.15, 1.0),
        elinewidth=0.6,
        capsize=2.5,
        capthick=0.5,
        zorder=4,
    )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(f"{grp}\n(n={len(sub)})", fontsize=9, color=colour)
    ax.grid(True, linestyle="--", linewidth=0.3, alpha=0.5)
    ax.set_axisbelow(True)

    if idx % ncols == 0:
        ax.set_ylabel(r"$G - R$", fontsize=10)
    if idx >= (nrows - 1) * ncols:
        ax.set_xlabel(r"$R - I$", fontsize=10)

# Hide unused panels
for idx in range(n_groups, len(axes3_flat)):
    axes3_flat[idx].set_visible(False)

fig3.suptitle(
    r"Colour-Colour Diagram $(G-R)$ vs $(R-I)$ — per group" + "\nFink/LSST — Deep Drilling Fields",
    fontsize=13,
    y=1.01,
)
plt.tight_layout()

out_panels_png = os.path.join(DIR_FIGS, "colour_colour_gr_ri_per_group.png")
fig3.savefig(out_panels_png, dpi=150, bbox_inches="tight")
print(f"Saved: {out_panels_png}")
plt.show()

## 10. Summary

This notebook produced:

| Output file | Description |
|---|---|
| `figs_FINK_BLOCK_LC_01_01_02/colour_colour_gr_ri.png/.pdf` | Combined colour-colour diagram (all groups, with error bars) |
| `figs_FINK_BLOCK_LC_01_01_02/magnitude_histograms_gri.png` | Magnitude distributions per band (g, r, i) |
| `figs_FINK_BLOCK_LC_01_01_02/colour_colour_gr_ri_per_group.png` | Split-panel diagram — one panel per classification group |

### Notes

- All fluxes from `objects_all.parquet` are in **nanojansky (nJy)**, converted to
  AB magnitudes via $m = -2.5\log_{10}(f_{\rm nJy}) + 31.4$.
- Colour uncertainties are propagated in quadrature from per-band magnitude uncertainties.
- Objects with negative or zero median flux in any of the g, r, i bands are excluded
  from the diagram (no physical AB magnitude can be defined).
- If `objects_all.parquet` does not contain explicit per-band columns, the notebook
  falls back automatically to `flatness_metrics.csv` (mean flux per band per object).
